# 🐍 Day 12 · Python Basics — Classes, Memory & SQLite (Persistence)

> **Goal of this notebook:** A chatbot remembers you by saving messages per **thread id**. Learn classes, `defaultdict`, `uuid` and SQLite to build memory that survives restarts.

**How to use it**
1. Click on a grey code cell and press **Shift + Enter** to run it.
2. Read the comments (lines that start with `#`) — they explain every step.
3. Change the values and run the cell again. Breaking things is how you learn! 🙂

📦 **First** do *Step 1 - Install today's packages* in [`00_START_HERE.md`](00_START_HERE.md).

✅ No API key needed for this notebook.

### 📋 What you will learn
1. A memory class
2. One memory per conversation with `defaultdict`
3. Unique ids with `uuid`
4. The `config` dictionary
5. SQLite: a database in one file
6. Snapshots and *time travel* with `deepcopy`

## 1. A memory class

In [ ]:
class ChatMemory:
    def __init__(self):
        self.messages = []                    # starts empty

    def add(self, role, content):
        self.messages.append({"role": role, "content": content})

    def last(self, n=2):
        return self.messages[-n:]             # the last n messages

memory = ChatMemory()
memory.add("user", "My name is Neha")
memory.add("assistant", "Nice to meet you, Neha!")
memory.add("user", "What is my name?")
print(memory.last(2))
print(len(memory.messages), "messages stored")

## 2. One memory per conversation
Each conversation has a **thread id**. `defaultdict(list)` automatically creates an empty list for a new id.

In [ ]:
from collections import defaultdict

threads = defaultdict(list)
threads["thread-1"].append("Hi, I'm Neha")
threads["thread-2"].append("Hi, I'm Omar")
threads["thread-1"].append("Remember my name?")

print(dict(threads))
print("Thread 3 is new:", threads["thread-3"])   # empty list created automatically

## 3. Unique ids with `uuid`

In [ ]:
import uuid
thread_id = str(uuid.uuid4())      # a random id that will never repeat
print(thread_id)

## 4. The `config` dictionary
In LangGraph you pass the thread id inside a nested dictionary:
```python
graph.invoke(state, config={"configurable": {"thread_id": "1"}})
```

In [ ]:
config = {"configurable": {"thread_id": thread_id}}
print(config["configurable"]["thread_id"])     # reading a nested dictionary

## 5. SQLite — a real database in a single file
`SqliteSaver` (used in `02_sqlite_saver.py`) stores checkpoints in SQLite. Here is SQLite on its own.
- `?` placeholders keep your queries safe.

In [ ]:
import sqlite3, os

conn = sqlite3.connect("demo_memory.db")      # creates the file if needed
conn.execute("CREATE TABLE IF NOT EXISTS messages (thread_id TEXT, role TEXT, content TEXT)")

conn.execute("INSERT INTO messages VALUES (?, ?, ?)", ("t1", "user", "My name is Neha"))
conn.execute("INSERT INTO messages VALUES (?, ?, ?)", ("t1", "assistant", "Hello Neha"))
conn.execute("INSERT INTO messages VALUES (?, ?, ?)", ("t2", "user", "I am Omar"))
conn.commit()                                  # save the changes
conn.close()

In [ ]:
# Open the file again — the data is still there (that is persistence!)
conn = sqlite3.connect("demo_memory.db")
rows = conn.execute("SELECT role, content FROM messages WHERE thread_id = ?", ("t1",)).fetchall()
for role, content in rows:
    print(role, ":", content)
conn.close()

os.remove("demo_memory.db")     # tidy up

## 6. Snapshots and time travel
LangGraph saves a **snapshot** of the state after every step, so you can go back. `deepcopy` makes a full independent copy.

In [ ]:
from copy import deepcopy

state = {"messages": []}
history = []                                   # all snapshots

for text in ["hi", "tell me a joke", "another one"]:
    state["messages"].append(text)
    history.append(deepcopy(state))            # copy, not a reference!

print("Now:       ", state)
print("Go back to step 1:", history[0])

---
## 🏋️ Practice time
Try these on your own in the empty cell below:

1. Add a method `clear()` to `ChatMemory` that empties the list.
2. Create a `config` dictionary for thread id `"student-42"` and print the id from it.
3. Insert one more row into a SQLite table and count the rows with `SELECT COUNT(*) FROM messages`.

In [ ]:
# ✍️ Write your answers here

<details>
<summary>👀 Click here to see the solutions (try first!)</summary>

```python
class ChatMemory2(ChatMemory):
    def clear(self):
        self.messages = []

cfg = {"configurable": {"thread_id": "student-42"}}
print(cfg["configurable"]["thread_id"])

conn = sqlite3.connect(":memory:")   # ':memory:' = database in RAM only
conn.execute("CREATE TABLE messages (role TEXT)")
conn.execute("INSERT INTO messages VALUES ('user')")
print(conn.execute("SELECT COUNT(*) FROM messages").fetchone()[0])
```
</details>

## ✅ Summary

| Concept | Example |
|---|---|
| Class with a list | `self.messages = []` |
| Auto-create values | `defaultdict(list)` |
| Unique id | `str(uuid.uuid4())` |
| Nested dict | `config['configurable']['thread_id']` |
| SQLite connect | `sqlite3.connect('a.db')` |
| Safe query | `conn.execute('... WHERE id = ?', (x,))` |
| Full copy | `deepcopy(state)` |

## ➡️ What's next?
You now know the Python used in **Day_12_LangGraph_Memory_and_Persistence**.

Open [`00_START_HERE.md`](00_START_HERE.md) and continue with **lesson 1**. 🚀